In [1]:
import pandas as pd
from openai import OpenAI
from autoddg import AutoDDG
from autoddg.utils import get_sample
from autoddg.evaluation import BaseEvaluator
from typing import Optional
# --- Import custom files ---
from prompts import ALL_RELATED_WORK_PROMPTS
from utils import log_result, run_description_experiment
import os 
import json
from cache_utils import run_with_caching, load_profile_from_cache#, MockAutoDDG           #Mock is for testing

[SETUP] Project Root: /Users/sara/Desktop/NYUFALL2025/NLP/FinalProject/AutoDDG-Enhanced
[SETUP] Cache Directory: /Users/sara/Desktop/NYUFALL2025/NLP/FinalProject/AutoDDG-Enhanced/prompt-experiments/profile_cache


In [10]:
import json
import pandas as pd

results = pd.read_csv("results.csv")
DATABASE_PATH_ = "../src/autoddg/database.json"

with open(DATABASE_PATH_, "r", encoding="utf-8") as f:
    db = json.load(f)

# dataset_name -> description lookup
name_to_desc = {
    v["dataset_name"].strip(): v.get("description", "")
    for v in db.values()
    if "dataset_name" in v
}

# add column (no overwriting other columns)
results["Reference_Description"] = (
    results["Dataset_Name"].astype(str).str.strip().map(name_to_desc)
)

# optional: keep blanks instead of NaN
results["Reference_Description"] = results["Reference_Description"].fillna("")

results.to_csv("results_refdesc.csv", index=False)


In [15]:
###----- COMPUTE ROUGE AND BERTSCORE ---- 

import pandas as pd
import evaluate

# ---- load ----
df = pd.read_csv("results_refdesc.csv")

# change these if your column names differ
REF_COL = "Reference_Description"
PRED_COL = "Description_Text"   # or "Generated_Description" etc.

# basic cleaning (avoid NaNs breaking metrics)
refs = df[REF_COL].fillna("").astype(str).tolist()
preds = df[PRED_COL].fillna("").astype(str).tolist()

# optional: drop rows where either side is empty (recommended)
mask = [(r.strip() != "" and p.strip() != "") for r, p in zip(refs, preds)]
df_eval = df.loc[mask].copy()
refs_eval = [r for r, m in zip(refs, mask) if m]
preds_eval = [p for p, m in zip(preds, mask) if m]

# ---- ROUGE ----
rouge = evaluate.load("rouge")
rouge_out = rouge.compute(predictions=preds_eval, references=refs_eval, use_stemmer=True)

# evaluate's ROUGE here is *aggregate*; we want per-row.
# So use rouge-score directly for per-example:
from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

r1, r2, rL = [], [], []
for p, r in zip(preds_eval, refs_eval):
    s = scorer.score(r, p)  # (target, prediction)
    r1.append(s["rouge1"].fmeasure)
    r2.append(s["rouge2"].fmeasure)
    rL.append(s["rougeL"].fmeasure)

df_eval["rouge1_f"] = r1
df_eval["rouge2_f"] = r2
df_eval["rougeL_f"] = rL

# ---- BERTScore ----
bertscore = evaluate.load("bertscore")
bs = bertscore.compute(
    predictions=preds_eval,
    references=refs_eval,
    lang="en",                 # change if not English
    model_type="microsoft/deberta-xlarge-mnli",  # strong default; switch if too slow
    rescale_with_baseline=True
)

df_eval["bertscore_p"] = bs["precision"]
df_eval["bertscore_r"] = bs["recall"]
df_eval["bertscore_f1"] = bs["f1"]

# ---- merge back (keep rows we skipped as NaN) ----
for col in ["rouge1_f", "rouge2_f", "rougeL_f", "bertscore_p", "bertscore_r", "bertscore_f1"]:
    df[col] = pd.NA
df.loc[df_eval.index, df_eval.columns.intersection(df.columns)] = df_eval

df.to_csv("results_with_rouge_bertscore.csv", index=False)
print("Wrote results_with_rouge_bertscore.csv")


Wrote results_with_rouge_bertscore.csv


In [ ]:
import pandas as pd


from coverage import coverage_bundle 

df = pd.read_csv("results_with_rouge_bertscore.csv")

out_rows = []
for i, row in df.iterrows():
    ref = row.get("Reference_Description", "")
    gen = row.get("Description_Text", "")

    # If you already computed BERTScore recall and saved it as a column:
    bsr = row.get("bertscore_r", None)

    scores = coverage_bundle(ref, gen, bertscore_recall=None if pd.isna(bsr) else float(bsr))
    out_rows.append(scores)

scores_df = pd.DataFrame(out_rows)
df = pd.concat([df, scores_df], axis=1)
df.to_csv("results_with_coverage.csv", index=False)
print("Wrote results_with_coverage.csv")


FileNotFoundError: [Errno 2] No such file or directory: 'results_with_rouge_bertscore.csv.csv'

In [21]:
! pip install sentence_transformers